# 14. Métricas de evaluación

Cálculo de las métricas del criterio de éxito sobre el pronóstico retrospectivo de la notebook 13 (tarea 16 del backlog, HU-05), el error del máximo estacional de cada modelo, la cobertura de la banda, el error de la proyección del cargo y el sobrecosto evitable.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from modelado import asignar_estaciones, calcular_maximos_estacionales, pronostico_anterior_mensual
from regulacion import calcular_piso, asignar_piso, calcular_hrp, cargo_escenarios, sobrecosto_evitable
from evaluacion import errores_maximo, cargos_retrospectivos, resumen_metricas

In [3]:
ruta_retrospectivo = Path("../data/processed/RETROSPECTIVO_CTRELEUW.csv")
ruta_retrospectivo_mensual = Path("../data/processed/RETROSPECTIVO_MENSUAL_CTRELEUW.csv")
retrospectivo_mensual = pd.read_csv(ruta_retrospectivo_mensual, parse_dates=["MES"])
retrospectivo = pd.read_csv(ruta_retrospectivo)

In [4]:
retrospectivo

,ESTACION,ANIO_ESTACION,MAXIMO_MW,PRONOSTICO_ANT,Q10_MW,Q50_MW,Q90_MW,HW_MW
0,INVIERNO,2021,42.352,NaN,45.414681,46.027273,47.260814,40.114391
1,INVIERNO,2022,44.156,42.352,43.734627,45.302328,46.840297,42.563225
2,INVIERNO,2023,42.448,44.156,43.181230,45.017425,46.953116,42.098513
3,INVIERNO,2024,41.960,42.448,42.678967,44.313533,46.699444,42.394445
4,INVIERNO,2025,44.436,41.960,42.253762,43.863804,46.760655,42.071078
5,VERANO,2021,44.380,44.396,43.781919,44.340467,46.007473,41.569720
6,VERANO,2022,45.528,44.380,43.314707,44.314921,45.871326,40.944850
7,VERANO,2023,48.340,45.528,42.831921,44.551764,45.881774,42.461331
8,VERANO,2024,47.856,48.340,42.829853,45.065222,46.901773,42.200645
9,VERANO,2025,44.244,47.856,42.697954,45.233164,47.076460,43.833393


## 1. Error del máximo estacional

Los tres modelos se comparan solo en las 9 estaciones donde los tres pronostican. Se excluye el invierno 2021, que no tiene pronóstico de la repetición del pico anterior. El error se calcula como real menos pronóstico.

In [5]:
sin_na = retrospectivo["PRONOSTICO_ANT"].notna()
comunes = retrospectivo[sin_na].reset_index(drop=True).copy()
len(comunes)

9

In [6]:
comunes["ERROR_RC"] = comunes["MAXIMO_MW"] - comunes["Q50_MW"]
comunes["ERROR_HW"] = comunes["MAXIMO_MW"] - comunes["HW_MW"]
comunes["ERROR_ANT"] = comunes["MAXIMO_MW"] - comunes["PRONOSTICO_ANT"]

In [7]:
mae_rc = comunes["ERROR_RC"].abs().mean()
mae_hw = comunes["ERROR_HW"].abs().mean()
mae_ant = comunes["ERROR_ANT"].abs().mean()

In [8]:
print("MAE regresión cuantílica: ", round(mae_rc, 3))
print("MAE Holt-Winters: ", round(mae_hw, 3))
print("MAE repetición del pico anterior: ", round(mae_ant, 3))

MAE regresión cuantílica:  1.718
MAE Holt-Winters:  2.676
MAE repetición del pico anterior:  1.616


### Error por tipo de estación

Se separan los errores de invierno y de verano, ya que un sesgo cercano a cero en el conjunto puede ocultar errores de signo opuesto.

In [9]:
errores_tipo = comunes.groupby("ESTACION")[["ERROR_RC", "ERROR_HW", "ERROR_ANT"]]
mae_tipo = errores_tipo.agg(lambda x: x.abs().mean()).round(2)
sesgo_tipo = errores_tipo.mean().round(2)
print("MAE por tipo de estación:\n", mae_tipo, "\n")
print("Sesgo por tipo de estación:\n", sesgo_tipo)

MAE por tipo de estación:
           ERROR_RC  ERROR_HW  ERROR_ANT
ESTACION                               
INVIERNO      1.66      1.19       1.62
VERANO        1.76      3.87       1.61 

Sesgo por tipo de estación:
           ERROR_RC  ERROR_HW  ERROR_ANT
ESTACION                               
INVIERNO     -1.37      0.97       0.52
VERANO        1.37      3.87      -0.03


## 2. Cobertura de la banda

Proporción de estaciones donde el máximo real quedó entre los cuantiles 0,1 y 0,9 del máximo pronosticado. La cobertura nominal es del 80 %.

In [10]:
real = retrospectivo["MAXIMO_MW"]

In [11]:
por_debajo = (real < retrospectivo["Q10_MW"]).mean()
por_encima = (real > retrospectivo["Q90_MW"]).mean()

In [12]:
cobertura = ((real >= retrospectivo["Q10_MW"]) & (real <= retrospectivo["Q90_MW"])).mean()

In [13]:
print("Cobertura: ", cobertura)
print("Por debajo del 0.1: ", por_debajo)
print("Por encima del 0.9: ", por_encima)

Cobertura:  0.5
Por debajo del 0.1:  0.3
Por encima del 0.9:  0.2


## 3. Error de la proyección del cargo

Para cada estación se calcula el cargo de referencia, el que resulta de la demanda registrada declarando el piso, y el cargo proyectado con los pronósticos mensuales de cada modelo, también declarando el piso. El error es la diferencia entre los dos.

Antes de noviembre 2025 el cargo es contrafactual. Se calcula con las reglas actuales, no con las que regían.

In [14]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie = pd.read_csv(ruta_serie, parse_dates=["MES"])

In [15]:
reales = serie[["MES", "MAX_REAL_MW"]]
retrospectivo_mensual = retrospectivo_mensual.merge(reales, on="MES", how="left")

In [16]:
retrospectivo_mensual = pronostico_anterior_mensual(serie, retrospectivo_mensual)

In [17]:
serie = asignar_estaciones(serie)
maximos = calcular_maximos_estacionales(serie)
maximos = calcular_piso(maximos)

In [18]:
retrospectivo_mensual = asignar_piso(retrospectivo_mensual, maximos)
retrospectivo_mensual.loc[retrospectivo_mensual["ANIO_ESTACION"].isin([2022, 2025]), ["ESTACION", "ANIO_ESTACION", "MES", "PISO_MW"]]

,ESTACION,ANIO_ESTACION,MES,PISO_MW
6,INVIERNO,2022,2022-05-01,38.117
7,INVIERNO,2022,2022-06-01,38.117
8,INVIERNO,2022,2022-07-01,38.117
9,INVIERNO,2022,2022-08-01,38.117
10,INVIERNO,2022,2022-09-01,38.117
11,INVIERNO,2022,2022-10-01,38.117
24,INVIERNO,2025,2025-05-01,37.764
25,INVIERNO,2025,2025-06-01,37.764
26,INVIERNO,2025,2025-07-01,37.764
27,INVIERNO,2025,2025-08-01,37.764


In [19]:
retrospectivo_mensual = calcular_hrp(retrospectivo_mensual)
retrospectivo_mensual.loc[retrospectivo_mensual["MES"] == "2025-11-01", ["MES", "HRP"]]

,MES,HRP
54,2025-11-01,370


In [20]:
escenarios = {"CARGO_REFERENCIA_USD": "MAX_REAL_MW", "CARGO_RC_USD": "PRED_RC_MW", "CARGO_HW_USD": "PRED_HW_MW", "CARGO_ANT_USD": "PRED_ANT_MW"}
retrospectivo_mensual = cargo_escenarios(retrospectivo_mensual, escenarios)
retrospectivo_mensual.loc[retrospectivo_mensual["MES"] == "2025-11-01", ["MAX_REAL_MW", "PRED_RC_MW", "PISO_MW", "HRP", "CARGO_REFERENCIA_USD", "CARGO_RC_USD"]]

,MAX_REAL_MW,PRED_RC_MW,PISO_MW,HRP,CARGO_REFERENCIA_USD,CARGO_RC_USD
54,38.628,40.666912,35.892,370,205809.98,216673.31


In [21]:
columnas_cargo = list(escenarios.keys())
cargos = retrospectivo_mensual.groupby(["ESTACION", "ANIO_ESTACION"])[columnas_cargo].sum(min_count=6).reset_index()
cargos

,ESTACION,ANIO_ESTACION,CARGO_REFERENCIA_USD,CARGO_RC_USD,CARGO_HW_USD,CARGO_ANT_USD
0,INVIERNO,2021,1522819.45,1660115.52,1468155.19,1537319.22
1,INVIERNO,2022,1561564.98,1615108.12,1549524.97,1524283.12
2,INVIERNO,2023,1561832.92,1570058.09,1542031.13,1577739.51
3,INVIERNO,2024,1509399.01,1554399.45,1546077.53,1551476.23
4,INVIERNO,2025,1567531.22,1542833.81,1535975.55,1503996.61
5,VERANO,2021,1542153.18,1555188.07,1502625.83,1556866.37
6,VERANO,2022,1575469.36,1527804.71,1492354.87,1534614.45
7,VERANO,2023,1614764.66,1563252.34,1547807.88,1595209.54
8,VERANO,2024,1649242.02,1618005.54,1618005.54,1663273.30
9,VERANO,2025,1580665.10,1532197.04,1520999.23,1531366.33


In [22]:
retrospectivo_mensual.loc[(retrospectivo_mensual["ESTACION"] == "VERANO") & (retrospectivo_mensual["ANIO_ESTACION"] == 2024), ["MES", "MAX_REAL_MW", "PRED_RC_MW", "PRED_HW_MW", "PRED_ANT_MW", "PISO_MW"]]

,MES,MAX_REAL_MW,PRED_RC_MW,PRED_HW_MW,PRED_ANT_MW,PISO_MW
48,2024-11-01,37.926,40.775220,37.960797,38.508,43.506
49,2024-12-01,40.441,41.870688,39.087672,37.004,43.506
50,2025-01-01,47.856,42.974850,41.923058,48.340,43.506
51,2025-02-01,40.894,42.323789,42.200645,45.172,43.506
52,2025-03-01,40.842,40.836117,38.243574,35.828,43.506
53,2025-04-01,37.499,38.415550,37.575607,38.104,43.506


In [23]:
for modelo in ["RC", "HW", "ANT"]:
    cargos[f"ERROR_{modelo}_USD"] = cargos["CARGO_REFERENCIA_USD"] - cargos[f"CARGO_{modelo}_USD"]
resumen_cargo = pd.DataFrame({
    "MAE_USD": cargos[["ERROR_RC_USD", "ERROR_HW_USD", "ERROR_ANT_USD"]].abs().mean(),
    "SESGO_USD": cargos[["ERROR_RC_USD", "ERROR_HW_USD", "ERROR_ANT_USD"]].mean()
}).round(2)
resumen_cargo


,MAE_USD,SESGO_USD
ERROR_RC_USD,46067.86,-5352.08
ERROR_HW_USD,43524.12,36188.42
ERROR_ANT_USD,31175.33,10929.72


## 4. Sobrecosto evitable

Se compara el cargo con lo efectivamente declarado contra el cargo que habría resultado de declarar el piso. Se calcula solo desde noviembre 2025, cuando empezó a regir la regla actual. El invierno 2026 tiene datos hasta septiembre, así que cubre cinco meses.

In [24]:
regimen_nuevo = sobrecosto_evitable(serie, maximos, "2025-11-01", "2026-10-01")
sobrecosto = regimen_nuevo.groupby(["ESTACION", "ANIO_ESTACION"])[["CARGO_DECLARADO_USD", "CARGO_PISO_USD", "SOBRECOSTO_USD"]].sum().reset_index()
sobrecosto

,ESTACION,ANIO_ESTACION,CARGO_DECLARADO_USD,CARGO_PISO_USD,SOBRECOSTO_USD
0,INVIERNO,2026,1435629.26,1375104.2,60525.06
1,VERANO,2025,1580665.10,1580665.1,0.00


## 5. Error del cargo por estación

In [25]:
cargos[["ESTACION", "ANIO_ESTACION", "ERROR_RC_USD"]]

,ESTACION,ANIO_ESTACION,ERROR_RC_USD
0,INVIERNO,2021,-137296.07
1,INVIERNO,2022,-53543.14
2,INVIERNO,2023,-8225.17
3,INVIERNO,2024,-45000.44
4,INVIERNO,2025,24697.41
5,VERANO,2021,-13034.89
6,VERANO,2022,47664.65
7,VERANO,2023,51512.32
8,VERANO,2024,31236.48
9,VERANO,2025,48468.06


## 6. Verificación del módulo de evaluación

Se recalculan las métricas con las funciones de `src/evaluacion.py`, que usa la pantalla de evaluación retrospectiva. Deben de coincidir con las calculadas paso a paso en esta notebook.

In [26]:
retrospectivo_original = pd.read_csv(ruta_retrospectivo)
mensual_original = pd.read_csv(ruta_retrospectivo_mensual, parse_dates=["MES"])
errores = errores_maximo(retrospectivo_original)
cargos_nuevos = cargos_retrospectivos(mensual_original, serie, maximos)
resumen_metricas(errores, cargos_nuevos).round(3)

,MODELO,MAE_MW,SESGO_MW,MAE_CARGO_USD
0,Regresión cuantílica,1.718,0.149,46067.863
1,Repetición del pico de la estación anterior,1.616,0.215,31175.332
2,Holt-Winters,2.676,2.579,43524.122


## Conclusión

- En el error del máximo estacional, la repetición del pico anterior (1,62 MW) supera levemente a la regresión cuantílica (1,72 MW), y las dos superan a Holt-Winters (2,68 MW), que pronostica de menos de forma sistemática (sesgo de +2,58 MW).
- El sesgo casi nulo de la regresión cuantílica en el conjunto oculta errores de signo opuesto: en invierno pronostica de más (sesgo de -1,37 MW) y en verano, de menos (+1,37 MW). En invierno, Holt-Winters tiene el menor error (1,19 MW), y en verano, la repetición del pico anterior (1,61 MW).
- La banda de la regresión cuantílica cubrió el 50 % de las estaciones, frente al 80 % nominal: un 30 % quedó por debajo y un 20 %, por encima. Es consistente con lo observado en la validación mensual: la banda del máximo resulta demasiado angosta.
- En la proyección del cargo, la repetición del pico anterior también tiene el menor error (31.175 dólares, frente a 46.068 de la regresión cuantílica y 43.524 de Holt-Winters). El piso reduce la diferencia entre los modelos, porque en los meses en que supera al pronóstico, el cargo no depende de este.
- Con estos resultados, la regresión cuantílica no cumple el criterio de éxito frente a la referencia.
- Las métricas se calcularon sobre pocas estaciones: las 10 reconstruidas entre 2021 y 2025, salvo el error del máximo, que excluye el invierno 2021 porque la repetición del pico anterior no tiene pronóstico para esa estación. Por eso, las diferencias entre los modelos son indicativas y no concluyentes.
- Declarar el piso en lugar de lo declarado habría evitado un sobrecosto de 60.525 dólares en el invierno 2026 (mayo a septiembre). En el verano 2025 la diferencia es nula, porque la demanda registrada superó lo declarado en todos los meses.